# Vision-Language Models

## 介绍

视觉语言模型（VLMs）打通了视觉感知与自然语言理解。传统计算机视觉模型仅输出数值标签或目标边界框，而视觉语言模型可用自然语言描述画面、针对图像内容作答，并依据文本描述定位图像中的目标。

VLM的发展建立在计算机视觉和自然语言处理的进步之上。2021年CLIP展示的大规模图像-文本对对比预训练取得了突破，表明经过图像-标题匹配训练的模型可以在数百个视觉任务上执行零样本分类，无需特定任务训练。

对于地理空间工作，VLM支持传统方法无法实现的任务。无需训练专门的分类器来检测游泳池，您只需向VLM询问"这张图片中有游泳池吗？"这种零样本能力大大降低了从遥感图像提取信息的门槛。

VLM有益于快速灾害损失评估、初步场地调查、环境监测和大规模土地利用清单等应用。分析师可以立即使用精心设计的提示从图像中提取见解，而无需等待自定义模型训练完成。

在本教程中，您将通过`geoai`库中的`MoondreamGeo`类使用[Moondream](https://moondream.ai)，这是一个针对边缘部署优化的轻量级开源VLM。您还将使用CLIP进行卫星图像的文本引导分割。

## 学习目标

在本教程结束时，您将能够：

- 理解视觉-语言模型如何结合视觉和文本推理
- 初始化和配置`MoondreamGeo`以进行可重复的地理空间分析
- 使用Moondream对卫星图像进行图像标题生成和视觉问答
- 使用文本引导提示在地理参考图像中检测和定位对象
- 使用交互式GUI进行探索性VLM分析
- 实现滑动窗口方法，将VLM应用于大型栅格并支持可配置的合并策略
- 执行基于CLIP的卫星图像零样本分割
- 评估VLM在地球观测中的优势和局限性

## 视觉-语言模型工作原理

视觉-语言模型通过在大规模图像-文本对数据集上训练来学习将图像与文本关联，将两种模态映射到共享的嵌入空间中，其中相关概念彼此接近。更多细节请参见这个[VLM解释器](https://huggingface.co/blog/vlms)。

**CLIP和对比学习。** CLIP（对比语言-图像预训练）使用图像编码器和文本编码器，训练目标是最大化匹配图像-文本对之间的相似度，同时最小化不匹配对之间的相似度。在数亿个网络来源的图像-文本对上训练后，CLIP获得了非常通用的视觉理解能力。

**从CLIP到生成式VLM。** 像Moondream这样的生成式VLM将视觉编码器（通常基于CLIP或SigLIP）与语言模型解码器相结合。视觉编码器将图像转换为特征token，与文本token一起输入语言模型以生成自由形式的文本响应。

**Moondream架构。** Moondream将SigLIP视觉编码器与Phi语言模型配对，总共约20亿参数。尽管规模较小，但它支持标题生成、视觉问答、目标检测和点定位。

**VLM对地理空间分析的重要性。** 传统遥感管道需要选择任务、收集标注数据、训练模型和部署。VLM将这一切简化为单一灵活的接口，同一模型可以通过不同提示生成标题、回答问题、检测对象和描述空间关系。

## 设置环境

安装所需的包并导入本教程中使用的库。

In [ ]:
# %pip install "geoai-py[extra]" transformers==4.57.6

Moondream v2与Transformers 5.x不兼容，请安装版本4.57.6或其他兼容的4.x版本。

In [ ]:
import geoai
import leafmap
from geoai import MoondreamGeo

## 样本数据

下载包含建筑物、车辆和植被的停车场地理参考航空图像。

In [ ]:
url = "https://data.source.coop/opengeos/geoai/parking-lot.tif"
image_path = geoai.download_file(url)

在交互式地图上可视化图像。

In [ ]:
m = leafmap.Map()
m.add_raster(image_path, layer_name="Satellite Image")
m

## 初始化Moondream处理器

`MoondreamGeo`类包装了Moondream模型并提供地理空间支持。固定`revision`日期将模型权重锁定到特定检查点以确保可重复性。首次运行会从[Hugging Face](https://huggingface.co/vikhyatk/moondream2)下载约3.8 GB。

In [ ]:
processor = MoondreamGeo(
    model_name="vikhyatk/moondream2",
    revision="2025-06-21",
)

使用GeoTIFF输入时，`MoondreamGeo`自动将像素坐标输出转换为地理坐标，返回结果作为可直接用于GIS工作流的GeoDataFrame。

## 图像标题生成

图像标题生成会生成图像内容的自然语言描述。`caption`方法接受`length`参数：`"short"`、`"normal"`或`"long"`。

In [ ]:
result = processor.caption(image_path, length="short")
print(result["caption"])

```text
城市停车场的鸟瞰图显示了两栋建筑、车辆和树木，以及复杂的道路和停车位网络。
```

In [ ]:
result = processor.caption(image_path, length="normal")
print(result["caption"])

```text
鸟瞰图展示了一个仓库式建筑群，大部分为浅灰色或白色，散布着几个停车场。一条宽阔的铺装道路贯穿中心，有多条车道。图像中左部可见一棵大树，在工业景观中增添了一抹绿色。停车场停满了汽车，表明该区域有活跃的人类活动。鸟瞰视角突出了建筑群的复杂布局。
```

In [ ]:
result = processor.caption(image_path, length="long")
print(result["caption"])

```text
这张航空图像显示了一个宽敞的停车场区域，周围环绕着两栋大型建筑。停车场被清晰标记的停车位、道路和人行道分成多个区域。停车场停满了整齐排列的汽车，主要是白色和红色车辆。建筑物有平坦的灰色屋顶，设计现代。该区域维护良好，停车场周围种植了绿色树木和灌木，为城市环境增添了一抹绿意。图像捕捉了停车场布局、停车状况和周边建筑的全貌，清晰展示了商业区和住宅区。
```

`"long"`标题识别了较短标题省略的特定特征。与任何生成模型一样，将标题视为描述性摘要而非真实情况。

## 视觉问答

视觉问答（VQA）允许您询问关于图像的特定问题并接收文本答案，这对于评估土地覆盖、计数特征或评估状况非常有用。

In [ ]:
result = processor.query("How many buildings are in the image?", image_path)
print(result["answer"])

```text
图像中有两栋建筑。
```

In [ ]:
result = processor.query("What are the building roof colors?", image_path)
print(result["answer"])

```text
建筑屋顶颜色是白色和红色。
```

In [ ]:
result = processor.query(
    "What types of vehicles are visible in the parking areas?", image_path
)
print(result["answer"])

```text
停车场包含各种类型的车辆，包括汽车、卡车和公交车。
```

VQA非常适合快速评估任务，例如自然灾害后查询航空照片。具体、范围明确的问题比模糊问题产生更可靠的答案，提供关于高度或比例的上下文有助于校准响应。

## 目标检测和点定位

Moondream可以根据文本提示在图像中定位对象。`detect`方法返回边界框，而`point`方法返回中心点坐标。两种方法都自动对GeoTIFF输入的结果进行地理参考。

### 检测建筑

In [ ]:
result = processor.detect(image_path, "building", output_path="buildings.geojson")
print(f"Detected {len(result['objects'])} buildings")

In [ ]:
result["gdf"]

将地理参考检测结果添加到地图。

In [ ]:
style = {"color": "red", "weight": 2}
m.add_gdf(result["gdf"], layer_name="Buildings", style=style)
m

### 定位建筑质心

`point`方法找到每个对象的中心点，这对于计数特征和分析空间分布很有用。

In [ ]:
result = processor.point(
    image_path, "building", output_path="building_centroids.geojson"
)
print(f"Found {len(result['points'])} building centroids")

In [ ]:
m.add_gdf(result["gdf"], layer_name="Building Centroids")
m

### 检测树木

应用相同的检测方法定位树木。

In [ ]:
result = processor.detect(image_path, "tree", output_path="trees.geojson")
print(f"Detected {len(result['objects'])} trees")

In [ ]:
m.add_gdf(result["gdf"], layer_name="Trees", style={"color": "green", "weight": 2})

### 定位树木质心

In [ ]:
result = processor.point(image_path, "tree", output_path="tree_centroids.geojson")
print(f"Found {len(result['points'])} tree centroids")

添加树木质心以一起查看所有检测图层。

In [ ]:
m.add_gdf(result["gdf"], layer_name="Tree Centroids")
m

检测对于计数特征和识别空间分布很有用。如需精确描绘，考虑将VLM检测作为SAM等分割模型的初始提示。

## 交互式GUI

`MoondreamGeo`类包含内置交互式GUI，无需为每个查询编写代码即可进行探索性分析。

In [ ]:
moondream = MoondreamGeo(
    model_name="vikhyatk/moondream2",
    revision="2025-06-21",
)
moondream.load_image(image_path)
m_gui = moondream.show_gui()
m_gui

GUI提供以下控件：

- **模式**：在标题、查询、检测和点分析模式之间切换
- **提示**：为查询、检测或点模式输入文本提示（例如"building"、"green tree"或"How many cars are parked?"）
- **长度**：使用标题模式时选择标题长度（短、正常、长）
- **透明度**：调整检测叠加层的透明度
- **颜色**：选择边界框和点标记的颜色
- **运行**：执行选定的操作并在地图上显示结果
- **保存**：将结果导出到GeoJSON文件
- **重置**：清除所有结果并恢复原始地图视图

检测结果显示为边界框，点结果显示为圆形标记，直接显示在地图上。运行操作后，您可以作为GeoDataFrame访问结果。

In [ ]:
gdf = m_gui.last_result_as_gdf
gdf

GUI对于探索图像、测试提示或向利益相关者演示VLM功能特别有用。

## 大型栅格的滑动窗口分析

卫星和航空图像通常太大，无法作为单个VLM输入处理。滑动窗口方法将栅格分成较小的重叠切片，独立处理每个切片，并聚合结果。

关键参数是：

- `window_size`：每个正方形切片的像素尺寸（默认：512）
- `overlap`：相邻切片之间重叠的像素数（默认：64）
- `iou_threshold`：合并重叠检测时用于非极大值抑制的交并比阈值（默认：0.5）
- `combine_strategy`：如何合并每个切片的文本结果用于查询和标题方法（`"concatenate"`或`"summarize"`）

### 滑动窗口目标检测

通过自动非极大值抑制（NMS）在所有切片上检测对象，以移除重叠区域的重复检测。

In [ ]:
result = processor.detect_sliding_window(
    image_path,
    "car",
    window_size=512,
    overlap=64,
    iou_threshold=0.5,
    output_path="cars_sliding_window.geojson",
)
print(f"Detected {len(result['objects'])} cars")

In [ ]:
result["gdf"].head()

在交互式地图上可视化滑动窗口车辆检测结果。

In [ ]:
m2 = leafmap.Map()
m2.add_raster(image_path, layer_name="Satellite Image")
m2.add_gdf(
    result["gdf"],
    layer_name="Detected Cars",
    style={"color": "red", "fillOpacity": 0.3},
)
m2

### 滑动窗口点检测

在大型图像上找到特定对象的位置作为点。

In [ ]:
trees = processor.point_sliding_window(
    image_path,
    "tree",
    window_size=512,
    overlap=64,
    output_path="trees_sliding_window.geojson",
)
print(f"Found {len(trees['points'])} tree locations")

在交互式地图上可视化检测到的树木位置。

In [ ]:
m3 = leafmap.Map()
m3.add_raster(image_path, layer_name="Satellite Image")
m3.add_gdf(trees["gdf"], layer_name="Trees", style={"color": "green", "radius": 3})
m3

### 视觉问答 with Sliding Window

逐个切片查询大型图像并合并每个切片的答案。`"concatenate"`策略将所有切片答案连接成单个字符串，保留所有信息但可能显得重复。

In [ ]:
result = processor.query_sliding_window(
    "What types of vehicles are visible?",
    image_path,
    window_size=512,
    overlap=64,
    combine_strategy="concatenate",
)
print(result["answer"])

```text
切片0（区域(0, 0, 512, 512)）：图像中可见的车辆包括汽车和卡车。
切片1（区域(448, 0, 960, 512)）：图像中可见的车辆包括汽车和卡车。
切片2（区域(896, 0, 1408, 512)）：图像中可见的车辆包括汽车和卡车。
切片3（区域(1344, 0, 1659, 512)）：图像中可见汽车。
切片4（区域(0, 448, 512, 938)）：图像中可见的车辆包括汽车和卡车。
切片5（区域(448, 448, 960, 938)）：图像中可见的车辆包括汽车和卡车。
切片6（区域(896, 448, 1408, 938)）：图像中可见的车辆包括汽车和卡车。
切片7（区域(1344, 448, 1659, 938)）：图像中可见汽车。
```

`"summarize"`策略进行额外的模型调用来将每个切片的答案提炼成单个连贯的响应，但需要额外的计算。

In [ ]:
result = processor.query_sliding_window(
    "Describe the land use and features in this area.",
    image_path,
    window_size=512,
    overlap=64,
    combine_strategy="summarize",
)
print(result["answer"])

```text
图像中的区域展示了多样化的土地利用和特征，突出了不同城市环境的适应性和功能性。停车场位于建筑物旁边，展示了一个组织良好的车辆停车管理系统。停车场分为多个区域，有些区域比其他区域有更多开放空间，确保车辆可以高效舒适地停放。停车场周围的树木增添了一抹绿色，增强了城市景观的整体美感。停车场看起来维护良好且组织有序，为车辆提供充足的停车位，可容纳各种类型和尺寸的汽车。
```

单个切片答案可用于空间分析。

In [ ]:
for tile in result["tile_answers"][:2]:  # Show first 2 tiles
    print(f"Tile {tile['tile_id']}: {tile['answer']}\n")

```text
切片0：该区域有一个大型停车场，停放着成排的汽车。停车场位于一栋建筑旁边，可能是商业或办公楼。停车场周围环绕着树木，为城市环境增添了自然元素。停车场看起来组织良好，有不同类型车辆的指定停车位。

切片1：图像中的区域似乎是一个有多个停车位的停车场，停满了各种汽车。停车场周围环绕着树木，为该区域营造了自然宜人的氛围。停车位排列成行和列，有些区域比其他区域有更多开放空间。汽车以各种角度和距离停放，占据了整个停车场。停车位的整体布局和排列表明这是一个组织良好且高效的车辆停放系统。
```

### 图像标题生成 with Sliding Window

通过为每个切片生成标题并合并结果来为大型图像生成综合标题。

In [ ]:
result = processor.caption_sliding_window(
    image_path,
    window_size=512,
    overlap=64,
    length="normal",
    combine_strategy="concatenate",
)
print(result["caption"])

In [ ]:
result = processor.caption_sliding_window(
    image_path,
    window_size=512,
    overlap=64,
    length="long",
    combine_strategy="summarize",
)
print(result["caption"])

In [ ]:
geoai.empty_cache()

### 便捷函数

对于不管理`MoondreamGeo`实例的一次性操作，geoai库公开了模块级便捷函数。

In [ ]:
from geoai import moondream_detect_sliding_window

result = moondream_detect_sliding_window(
    image_path,
    "car",
    window_size=512,
    overlap=64,
    model_name="vikhyatk/moondream2",
    revision="2025-06-21",
)
print(f"Detected {len(result['objects'])} cars")

In [ ]:
geoai.view_vector_interactive(result["gdf"], tiles=image_path)

### 常规检测与滑动窗口检测比较

一次性处理完整图像与逐个切片处理通常会产生不同的检测数量，特别是对于小物体或密集排列的物体。

In [ ]:
processor = MoondreamGeo(
    model_name="vikhyatk/moondream2",
    revision="2025-06-21",
)

In [ ]:
regular_result = processor.detect(image_path, "car")
print(f"Regular detection: {len(regular_result['objects'])} cars")

sliding_result = processor.detect_sliding_window(
    image_path, "car", window_size=512, overlap=64
)
print(f"Sliding window detection: {len(sliding_result['objects'])} cars")

```text
常规检测：50辆汽车
滑动窗口检测：236辆汽车
```

滑动窗口方法找到更多汽车，因为小型车辆在每个切片内占据更多像素。完整图像推理通常足以检测建筑物等大物体，而滑动窗口对于车辆等小特征更可靠。

In [ ]:
geoai.empty_cache()

### 性能提示

调整滑动窗口参数时，考虑这些权衡：

- **窗口大小**：较小的切片（256-512像素）改善小物体的检测但增加处理时间。较大的切片（512-1024像素）更快但可能错过细节。
- **重叠**：较大的重叠（64-128像素）减少切片边界附近遗漏的物体但增加切片数量。较小的重叠（32-64像素）更快但可能遗漏边界物体。
- **IoU阈值**（仅限检测）：较高的阈值（0.6-0.8）保留更多检测但可能有重复。较低的阈值（0.3-0.5）合并更激进可能丢弃真正的附近物体。
- **合并策略**：使用`"concatenate"`以提高速度或进行逐切片检查。需要连贯摘要时使用`"summarize"`。

## 基于CLIP的分割

CLIPSeg从文本提示生成分割掩码，无需边界框或点提示，生成软概率图，其中较高的值表示与文本描述更强的对齐。

In [ ]:
clip_image_url = "https://data.source.coop/opengeos/geoai/uc-berkeley.tif"
clip_image_path = geoai.download_file(clip_image_url)

在交互式地图上可视化卫星图像。

In [ ]:
geoai.view_raster(clip_image_path)

初始化CLIPSeg模型，切片大小为512像素，重叠32像素。

In [ ]:
segmenter = geoai.CLIPSegmentation(tile_size=512, overlap=32)

定义分割的输出路径和文本提示。

In [ ]:
mask_output_path = "tree_masks.tif"
text_prompt = "trees"

以0.5的概率阈值和高斯平滑运行分割。

In [ ]:
segmenter.segment_image(
    clip_image_path,
    output_path=mask_output_path,
    text_prompt=text_prompt,
    threshold=0.5,
    smoothing_sigma=1.0,
)

可视化叠加在原始卫星图像上的树木掩码。

In [ ]:
geoai.view_raster(
    mask_output_path,
    nodata=0,
    opacity=0.7,
    colormap="greens",
    layer_name="Trees",
    basemap=clip_image_path,
)

创建分割地图以比较分割结果与原始图像。

In [ ]:
geoai.create_split_map(
    left_layer=mask_output_path,
    right_layer=clip_image_path,
    left_label="Trees",
    right_label="Satellite Image",
    left_args={"nodata": 0, "opacity": 0.8, "colormap": "greens"},
    basemap=clip_image_path,
)

基于CLIP的分割在没有训练数据时对于快速土地覆盖制图非常有用。软概率输出可以通过阈值化生成二值掩码，或跨多个提示组合用于多类别制图，尽管结果通常不如监督方法准确。

## 地球观测中的实际应用

VLM正在地理空间领域的多个应用中得到应用。

**灾害损失评估。** VLM可以处理数千张灾后航空图像，通过"这栋建筑的屋顶完好吗？"等查询来确定需要详细评估的区域优先级。

**环境监测。** VLM可以通过适当的提示帮助识别非法采矿、检测森林砍伐、评估湿地健康状况和监测海岸侵蚀。

**城市分析。** 城市规划师可以使用VLM大规模描述城市环境，回答关于数千个图像切片上的建筑密度、道路状况和绿地可用性的问题。

**农业监测。** VLM可以描述作物状况、识别田地边界并检测灌溉基础设施，无需特定作物的训练数据。

## 局限性和注意事项

**空间分辨率和尺度。** VLM可能难以处理卫星图像中的鸟瞰视角，特别是在高海拔地区，特征仅占据几个像素。

**幻觉。** VLM可能产生自信但不正确的响应，描述不存在的特征或错误计数对象。始终根据参考数据验证输出。

**空间精度。** 边界框和点坐标是近似的，不应视为测量级精度。

**一致性。** VLM在不同运行中可能对相同输入产生不同输出。具有约束答案格式的结构化提示产生更可靠的结果。

**提示敏感性。** 措辞的微小变化可能产生显著不同的响应。开发标准化提示模板并根据真实数据进行验证对于业务工作流很重要。

**领域差距。** 大多数VLM是在网络图像而非遥感数据集上训练的，因此性能可能因地区、季节和传感器类型而异。

## 关键要点

1. VLM连接视觉感知和自然语言，支持地理空间图像的灵活零样本分析。
2. CLIP通过对比学习学习共享的图像-文本表示，构成许多VLM架构的基础。
3. `MoondreamGeo`包装Moondream并提供地理空间支持，自动将输出地理参考为GeoDataFrame。
4. Moondream通过一致的API提供标题生成、VQA、目标检测和点定位。
5. 交互式GUI支持探索性分析，无需编写额外代码。
6. 滑动窗口方法将VLM功能扩展到任意大的栅格，支持可配置的合并策略。
7. CLIPSeg生成文本引导的分割掩码，无需训练数据即可快速进行土地覆盖制图。
8. 由于潜在的幻觉、有限的空间精度和尺度敏感性，VLM输出应进行验证。
9. 具体、结构化的提示比开放式查询产生更可靠的答案。